# 02 — Exploratory Data Analysis

Цель ноутбука: проверить пропуски, распределения, корреляции и связи между ключевыми признаками и success label.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

Project root: /Users/andrey/Documents/projects/COMPASS-AI
Synthetic data: /Users/andrey/Documents/projects/COMPASS-AI/data/synthetic


In [2]:
employees = pd.read_csv(DATA_SYNTHETIC / "employees.csv")
tasks = pd.read_csv(DATA_SYNTHETIC / "tasks.csv")
assignments = pd.read_csv(DATA_SYNTHETIC / "assignments.csv")

merged = assignments.merge(tasks, on="task_id", suffixes=("_assignment", "_task"))
merged = merged.merge(employees, on="employee_id", suffixes=("", "_employee"))

print("merged:", merged.shape)

merged: (60000, 60)


In [3]:
missing_report = pd.DataFrame(
    {
        "employees_missing": employees.isna().sum(),
        "tasks_missing": tasks.isna().sum(),
        "assignments_missing": assignments.isna().sum(),
    }
).fillna(0)

display(missing_report.sort_values("assignments_missing", ascending=False).head(30))

,employees_missing,tasks_missing,assignments_missing
plane_issue_id,0.0,2500.0,60000.0
plane_work_item_id,0.0,2500.0,60000.0
completed_at,0.0,0.0,4436.0
active_tasks_count,0.0,0.0,0.0
required_stack,0.0,0.0,0.0
name,0.0,0.0,0.0
outcome_status,0.0,0.0,0.0
plane_project_id,0.0,0.0,0.0
plane_user_id,36.0,0.0,0.0
primary_stack,0.0,0.0,0.0


In [4]:
numeric_cols = [
    "skill_match_score",
    "growth_match_score",
    "risk_score",
    "quality_score",
    "employee_workload_at_assignment",
    "delay_days",
    "success_probability",
    "success_label",
]

corr = assignments[numeric_cols].corr(numeric_only=True)
display(corr)

,skill_match_score,growth_match_score,risk_score,quality_score,employee_workload_at_assignment,delay_days,success_probability,success_label
skill_match_score,1.000000,0.257573,-0.714044,0.508720,-0.028284,-0.426501,0.791621,0.510922
growth_match_score,0.257573,1.000000,-0.217940,0.152210,0.090262,-0.137024,0.219364,0.164005
risk_score,-0.714044,-0.217940,1.000000,-0.548782,0.373325,0.444020,-0.867393,-0.575419
quality_score,0.508720,0.152210,-0.548782,1.000000,-0.146516,-0.760692,0.615220,0.765570
employee_workload_at_assignment,-0.028284,0.090262,0.373325,-0.146516,1.000000,0.109100,-0.236742,-0.161673
delay_days,-0.426501,-0.137024,0.444020,-0.760692,0.109100,1.000000,-0.500149,-0.578235
success_probability,0.791621,0.219364,-0.867393,0.615220,-0.236742,-0.500149,1.000000,0.651306
success_label,0.510922,0.164005,-0.575419,0.765570,-0.161673,-0.578235,0.651306,1.000000


In [5]:
fig = px.imshow(corr, text_auto=True, title="Correlation matrix")
fig.show()
fig.write_html(FIGURES / "eda_correlation_matrix.html")

In [6]:
success_by_skill = assignments.groupby("success_label")["skill_match_score"].mean().reset_index()

fig = px.bar(
    success_by_skill,
    x="success_label",
    y="skill_match_score",
    title="Average skill match by success label",
)
fig.show()
fig.write_html(FIGURES / "eda_skill_match_by_success.html")

In [7]:
success_by_workload = (
    assignments.groupby("success_label")["employee_workload_at_assignment"]
    .mean()
    .reset_index()
)

fig = px.bar(
    success_by_workload,
    x="success_label",
    y="employee_workload_at_assignment",
    title="Average workload by success label",
)
fig.show()
fig.write_html(FIGURES / "eda_workload_by_success.html")

In [8]:
complexity_success = merged.groupby("complexity")["success_label"].mean().reset_index()

fig = px.line(
    complexity_success,
    x="complexity",
    y="success_label",
    markers=True,
    title="Success rate by task complexity",
)
fig.show()
fig.write_html(FIGURES / "eda_success_by_complexity.html")